In [144]:

import pandas as pd
import os
import yaml
import logging
import sys
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score , recall_score , f1_score 



In [145]:
df = pd.read_csv("https://raw.githubusercontent.com/campusx-official/"
            "jupyter-masterclass/refs/heads/main/tweet_emotions.csv"
        )

In [146]:
df.head()

,tweet_id,sentiment,content
0,1956967341,empty,@tiffanylue i know i was listenin to bad habi...
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
3,1956967789,enthusiasm,wants to hang out with friends SOON!
4,1956968416,neutral,@dannycastillo We want to trade with someone w...


In [147]:
df.shape

(40000, 3)

In [148]:
x = df["sentiment"].isin(["happiness" , "sadness"])
df = df[x]

In [149]:
df.head()

,tweet_id,sentiment,content
1,1956967666,sadness,Layin n bed with a headache ughhhh...waitin o...
2,1956967696,sadness,Funeral ceremony...gloomy friday...
6,1956968487,sadness,"I should be sleep, but im not! thinking about ..."
8,1956969035,sadness,@charviray Charlene my love. I miss you
9,1956969172,sadness,@kelcouch I'm sorry at least it's Friday?


In [150]:
df.shape

(10374, 3)

In [151]:
import re 
import nltk
import string 
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download("WordNetLemmatizer")

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Error loading WordNetLemmatizer: Package
[nltk_data]     'WordNetLemmatizer' not found in index


False

In [152]:

import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Download required NLTK resources
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

# Initialize tools
stop_words = set(stopwords.words("english"))

# Keep negation words because they affect sentiment
stop_words = stop_words - {"no", "not", "nor"}

lemmatizer = WordNetLemmatizer()


# 1. Convert text to lowercase
def lower_case(text):
    return str(text).lower()


# 2. Remove URLs
def remove_urls(text):
    return re.sub(r"https?://\S+|www\.\S+", "", text)


# 3. Remove punctuation
def remove_punctuation(text):
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()


# 4. Remove numbers
def remove_number(text):
    return re.sub(r"\d+", "", text)


# 5. Remove non-ASCII characters
def remove_non_ascii(text):
    return re.sub(r"[^\x00-\x7F]+", "", text)


# 6. Remove stopwords
def remove_stop_word(text):
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)


# 7. Lemmatize words
def lemmatization(text):
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)


# 8. Complete text normalization pipeline
def normalization_sentence(text):
    text = lower_case(text)
    text = remove_urls(text)
    text = remove_punctuation(text)
    text = remove_number(text)
    text = remove_non_ascii(text)
    text = remove_stop_word(text)
    text = lemmatization(text)

    return text


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [153]:
df["content"] = df["content"].apply(normalization_sentence)

df["content"].head()

1               layin n bed headache ughhhhwaitin call
2                        funeral ceremonygloomy friday
6    sleep im not thinking old friend want he marri...
8                         charviray charlene love miss
9                       kelcouch im sorry least friday
Name: content, dtype: str

In [154]:
df.drop(columns =["tweet_id"] , inplace = True)

In [155]:
df.head()

,sentiment,content
1,sadness,layin n bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
6,sadness,sleep im not thinking old friend want he marri...
8,sadness,charviray charlene love miss
9,sadness,kelcouch im sorry least friday


In [156]:
df["sentiment"] = df["sentiment"].replace({"sadness":0 , "happiness":1} , inplace = True)
df.sample(5)

C:\Users\Hp\AppData\Local\Temp\ipykernel_22904\63258512.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df["sentiment"] = df["sentiment"].replace({"sadness":0 , "happiness":1} , inplace = True)


,sentiment,content
13605,0,therankinfiles fair asking mktg cd etc suggest...
35024,0,aplusk awww herewaz uuuup seriously glad yall ...
2785,0,last night bottle wine house cocktail came hom...
6834,1,missshell eeeeep jealous im work um u receive ...
2007,0,heading home could really done without banging...


In [157]:
tfidf_vectorizer = TfidfVectorizer(max_features = 1000)
x = tfidf_vectorizer.fit_transform(df["content"])
y = df["sentiment"]

In [158]:
X_train, X_test , Y_train , Y_test = train_test_split( x, y , test_size = 0.2 , random_state  = 42 )


In [159]:
import dagshub 
import mlflow 

dagshub.init(repo_owner='prasadshinde-2005', repo_name='Mini-project', mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/prasadshinde-2005/Mini-project.mlflow")


mlflow.set_experiment("Logistic_reg Baseline model_using_tfidf") 


Initialized MLflow to track repo "prasadshinde-2005/Mini-project"

Repository prasadshinde-2005/Mini-project initialized!

2026/10/10 19:19:43 INFO mlflow.tracking.fluent: Experiment with name 'Logistic_reg Baseline model_using_tfidf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/dc07d5645444465fb345cdffaad52c44', creation_time=1791640185328, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1791640185328, lifecycle_stage='active', name='Logistic_reg Baseline model_using_tfidf', tags={}, trace_location=None, workspace='default'>

In [160]:
df["sentiment"].value_counts()

sentiment
1    5209
0    5165
Name: count, dtype: int64

In [161]:
print("X_train shape:", X_train.shape)
print("Y_train shape:", Y_train.shape)

print("X_train type:", type(X_train))
print("Y_train type:", type(Y_train))

print("Missing values in Y_train:", Y_train.isna().sum())
print("Target distribution:\n", Y_train.value_counts())

X_train shape: (8299, 1000)
Y_train shape: (8299,)
X_train type: <class 'scipy.sparse._csr.csr_matrix'>
Y_train type: <class 'pandas.Series'>
Missing values in Y_train: 0
Target distribution:
 sentiment
1    4194
0    4105
Name: count, dtype: int64


In [162]:

print(pd.isna(Y_train).sum())

0


In [163]:
print("X train shape:", X_train.shape)
print("y train shape:", Y_train.shape)

print("X type:", type(X_train))
print("y type:", type(Y_train))

print("Missing values in y:", Y_train.isna().sum())
print("Target classes:", Y_train.value_counts())

X train shape: (8299, 1000)
y train shape: (8299,)
X type: <class 'scipy.sparse._csr.csr_matrix'>
y type: <class 'pandas.Series'>
Missing values in y: 0
Target classes: sentiment
1    4194
0    4105
Name: count, dtype: int64


Using mlflow doing experiment tracking 

In [164]:
Y_train = Y_train.astype(int)
Y_test = Y_test.astype(int)

print(Y_train.dtype) 

int64


In [165]:
with mlflow.start_run():

    # Log preprocessing parameter 

    mlflow.log_param("vectorizer" , "Bag of Words")
    mlflow.log_param("num_fetaures" , 1000)
    mlflow.log_param("test_size" , 0.2)

    # model training 
    model = LogisticRegression(max_iter = 1000)
    model.fit(X_train , Y_train)

    # log model parameters
    mlflow.log_param("model" ,"LogisticRegression")

    # model validation 
    y_pred = model.predict(X_test)

    # find accuracy 
    accuracy = accuracy_score(Y_test , y_pred)

    precision = precision_score(
        Y_test, y_pred, average="weighted", zero_division=0
    )
    recall = recall_score(
        Y_test, y_pred, average="weighted", zero_division=0
    )
    f1 = f1_score(
        Y_test, y_pred, average="weighted", zero_division=0
    )


    # log this evalution  metrics
    mlflow.log_metric("accuracy" , accuracy)
    mlflow.log_metric("precision" , precision)
    mlflow.log_metric("recall" , recall)
    mlflow.log_metric("f1" , f1)

     
    # print all the metrics 
    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 score:", f1)
    

Accuracy: 0.7956626506024096
Precision: 0.79676437175936
Recall: 0.7956626506024096
F1 score: 0.7956334155100189
🏃 View run angry-crow-2 at: https://dagshub.com/prasadshinde-2005/Mini-project.mlflow/#/experiments/1/runs/86a53121b89d48aaabef82b397286c73
🧪 View experiment at: https://dagshub.com/prasadshinde-2005/Mini-project.mlflow/#/experiments/1
